In [49]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [50]:
import pandas as pd
from CP2.customer_cleaner import CustomerCleaner

df=pd.read_csv("../../Dataset/customer_churn.csv")

customerclean=CustomerCleaner(data_frame=df)
df=customerclean.clean()



In [51]:
# 27

bins=[0,12,24,48,72]
labels=["0-12","13-24","25-48","49-72"]
df["tenure_bucket"]=pd.cut(
    df["tenure"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [52]:
df["tenure_bucket"].dtypes

CategoricalDtype(categories=['0-12', '13-24', '25-48', '49-72'], ordered=True, categories_dtype=str)

In [53]:
#28

median=df["monthly_charges"].mean()
def fill_high_charge(charge):
    return 1 if charge > median else 0

df["high_charge_flag"]=df["monthly_charges"].apply(fill_high_charge)

In [54]:
#29


def push_rows(value):
    service=["online_security", 
             "online_backup", 
             "device_protection", 
             "tech_support", 
             "streaming_tv", 
            "streaming_movies"] 
    count=0
    
    for cols in service:
        count += 1 if value[cols]== "Yes" else 0 
    return count
    
df["service_count"]=df.apply(push_rows,axis=1)

In [55]:
df["service_count"]


0       1
1       2
2       2
3       3
4       0
       ..
7038    5
7039    4
7040    1
7041    0
7042    5
Name: service_count, Length: 7043, dtype: int64

In [56]:
count=0
def rows(value):
    if value >= 24:
        return 1
    return 0


df["is_long_term_customer"]=df["tenure"].apply(rows)


In [57]:
df["is_long_term_customer"].head()

0    0
1    1
2    0
3    1
4    0
Name: is_long_term_customer, dtype: int64

In [58]:
### 31

def fill_streaming_bundle(value):
    if  value["streaming_tv"]=="Yes" and value["streaming_movies"]=="Yes" :
        return 1
    return 0

df["has_streaming_bundle"]=df.apply(fill_streaming_bundle,axis=1)

In [59]:
df["has_streaming_bundle"]

0       0
1       0
2       0
3       0
4       0
       ..
7038    1
7039    1
7040    0
7041    0
7042    1
Name: has_streaming_bundle, Length: 7043, dtype: int64

In [60]:
###32

def fill_payment(value):
    return 1 if "automatic" in value else 0 

df["auto_pay_flag"]=df["payment_method"].apply(fill_payment)

In [61]:
df["auto_pay_flag"]

0       0
1       0
2       0
3       1
4       0
       ..
7038    0
7039    1
7040    0
7041    0
7042    1
Name: auto_pay_flag, Length: 7043, dtype: int64

In [62]:
cols=["high_charge_flag","service_count","is_long_term_customer","has_streaming_bundle","auto_pay_flag","churn"]
df[cols].corr()

,high_charge_flag,service_count,is_long_term_customer,has_streaming_bundle,auto_pay_flag,churn
high_charge_flag,1.000000,0.531316,0.140786,0.428299,0.042787,0.203245
service_count,0.531316,1.000000,0.415006,0.678542,0.207668,-0.087698
is_long_term_customer,0.140786,0.415006,1.000000,0.262556,0.334694,-0.311496
has_streaming_bundle,0.428299,0.678542,0.262556,1.000000,0.077865,0.040441
auto_pay_flag,0.042787,0.207668,0.334694,0.077865,1.000000,-0.209902
churn,0.203245,-0.087698,-0.311496,0.040441,-0.209902,1.000000


In [63]:
df.to_csv("../../Dataset/customer_churn.csv")